# IMDB Data Pipeline — Ilesh

**End-to-end**: raw CSV shards → DuckDB quality fixes → PySpark feature engineering → LogisticRegression → predictions.

| Tool | Stage | Why here (not elsewhere) |
|---|---|---|
| **DuckDB** | Ingestion, 13 DQ checks, 9 fixes, success-rate encoding, Parquet export | In-process SQL; no cluster overhead for 8K rows. `TRY_CAST`, `QUANTILE_CONT`, UDF registration, `COPY TO PARQUET` — each check = one SQL string → trivial to update if schema changes. |
| **PySpark** | Load Parquet, MLlib Imputer + VectorAssembler + StandardScaler | MLlib `Imputer` is a Pipeline stage: fit on train, transform val/test with same medians → standard leakage-free distributed pattern. `StandardScaler` serialisable pipeline = critical for LR/SVM. |
| **sklearn** | LogisticRegression, predictions | Fast, familiar API. Features already scaled by Spark pipeline — no extra preprocessing. |
| **`re`/`unicodedata`** | Title normalisation (used as DuckDB UDF) | NFKD strips synthetically-injected accents (`Déstiny→Destiny`) in one pass; registered as DuckDB UDF so it runs inside SQL. |

**Sections**: 1 Setup · 2 Diagram · 3 Step1-DuckDB · 4 Step2-PySpark · 5 Step3-Model · 6 Reusability

---
## 1. Setup

**Docker**: the Dockerfile copies `config/`, `src/`, `data/` into `/app`.  
Mount `members/` and `submissions/` at runtime:
```bash
docker run -it --rm \
  -v $(pwd)/data:/app/data \
  -v $(pwd)/members:/app/members \
  -v $(pwd)/submissions:/app/submissions \
  big-data-imdb \
  python members/ilesh/pipeline.py
```

In [ ]:
import sys, re, unicodedata, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
warnings.filterwarnings("ignore")

# Resolve project root — works locally, in Docker (/app), and from any cwd
PROJECT_ROOT = Path.cwd().resolve()
for _ in range(8):
    if (PROJECT_ROOT / "config" / "config.yaml").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_CSV   = PROJECT_ROOT / "data" / "raw" / "csv"
OUT_DIR   = PROJECT_ROOT / "data" / "processed" / "ilesh"
PQDIR     = OUT_DIR / "parquet_base"
FEAT_DIR  = OUT_DIR / "features"
SUB_DIR   = PROJECT_ROOT / "submissions"
DB_PATH   = OUT_DIR / "step1.duckdb"
RPT_PATH  = OUT_DIR / "dq_report.txt"

for d in [PQDIR, FEAT_DIR, SUB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DIRECTORS_CSV = str(RAW_CSV / "movie_directors.csv")
WRITERS_CSV   = str(RAW_CSV / "movie_writers.csv")
TRAIN_GLOB    = str(RAW_CSV / "train-*.csv")
VAL_CSV       = str(RAW_CSV / "validation_hidden.csv")
TEST_CSV      = str(RAW_CSV / "test_hidden.csv")

print("Project root :", PROJECT_ROOT)
print("DuckDB       :", DB_PATH)

In [ ]:
# ── Title normalisation (Python + DuckDB UDF) ─────────────────────────────────
_STRIP_PUNCT = re.compile(r"[^\w\s]")
_COLLAPSE_WS = re.compile(r"\s+")

def _normalize_title(s: str) -> str:
    """NFKD + ASCII: strips synthetically-injected accents (Déstiny→Destiny).
    Safe for legitimate foreign titles (Le Samouraï → Le Samourai)."""
    if not s or s != s:
        return s
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = _STRIP_PUNCT.sub(" ", s)
    s = _COLLAPSE_WS.sub(" ", s).strip()
    return s

# Report log
_log = []
def rpt(msg=""):
    print(msg); _log.append(str(msg))

print("Helpers ready.")

---
## 2. Pipeline Diagram

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(15, 9))
ax.set_xlim(0, 15); ax.set_ylim(0, 9); ax.axis("off")
fig.patch.set_facecolor("#f5f5f5")

C = dict(DuckDB="#FFD700", PySpark="#E87722", sklearn="#32CD32",
         Regex="#DA70D6", Output="#FF6347", Data="#D0D0D0")

def box(x,y,w,h,lbl,sub,col,fs=8):
    ax.add_patch(mpatches.FancyBboxPatch((x-w/2,y-h/2),w,h,
        boxstyle="round,pad=0.07",linewidth=1.1,edgecolor="#444",
        facecolor=col,alpha=0.9,zorder=3))
    ax.text(x,y+0.12,lbl,ha="center",va="center",fontsize=fs,fontweight="bold",zorder=4)
    ax.text(x,y-0.20,sub,ha="center",va="center",fontsize=6.0,style="italic",color="#333",zorder=4)

def arr(x1,y1,x2,y2):
    ax.annotate("",xy=(x2,y2),xytext=(x1,y1),
        arrowprops=dict(arrowstyle="->",color="#555",lw=1.3),zorder=2)

# ── Data sources ──────────────────────────────────────────────────────────────
box(2.3,8.4,2.6,0.55, "train-*.csv (×8)",     "all_varchar=True glob",    C["Data"])
box(6.5,8.4,2.4,0.55, "directors/writers.csv","M:N edges",                C["Data"])
box(11.5,8.4,2.4,0.55,"val & test hidden",    "unlabelled splits",        C["Data"])

# ── STEP 1: DuckDB ───────────────────────────────────────────────────────────
box(2.3,7.3,2.6,0.6,  "D1–D13 DQ Checks",    "DuckDB SQL queries",       C["DuckDB"])
box(2.3,6.2,2.6,0.6,  "F1 Drop Unnamed:0",   "spurious pandas index",    C["DuckDB"])
box(2.3,5.1,2.6,0.6,  "F2 Drop endYear",     "90.1% missing, no signal", C["DuckDB"])
box(2.3,4.0,2.6,0.6,  "F3 \\N→NULL cast",    "TRY_CAST+NULLIF+TRIM",     C["DuckDB"])
box(2.3,2.9,2.6,0.6,  "F4 origTitle fill",   "COALESCE→primaryTitle",    C["DuckDB"])
box(2.3,1.8,2.6,0.6,  "F5 numVotes IQR cap", "train fence→all splits",   C["DuckDB"])
box(2.3,0.75,2.6,0.55,"F9 NFKD UDF",         "DuckDB UDF normalize_title",C["Regex"])

# ── M:N branch ────────────────────────────────────────────────────────────────
box(6.5,7.3,2.4,0.6,  "F6 Drop \\N IDs",     "at ingestion (2+297 rows)",C["DuckDB"])
box(6.5,6.2,2.4,0.6,  "F7 Success Rates",    "INNER JOIN train labels",  C["DuckDB"])

# ── Parquet handoff ───────────────────────────────────────────────────────────
box(4.5,0.75,2.2,0.55,"Export Parquet (×7)", "DuckDB COPY TO PARQUET",   C["DuckDB"])

# ── STEP 2: PySpark ──────────────────────────────────────────────────────────
box(7.5,4.0,2.4,0.6,  "Missingness Flags",   "PySpark: before impute",   C["PySpark"])
box(7.5,2.9,2.4,0.6,  "M:M LEFT JOIN agg",   "countDistinct avg/max",    C["PySpark"])
box(7.5,1.8,2.4,0.6,  "MLlib Imputer",        "median fit train-only",    C["PySpark"])
box(7.5,0.75,2.4,0.55,"log1p + VecAssembler","StandardScaler pipeline",  C["PySpark"])

# ── STEP 3: sklearn ───────────────────────────────────────────────────────────
box(11.5,2.9,2.4,0.6, "LogisticRegression",  "scaled features → fit",    C["sklearn"])
box(11.5,1.8,2.4,0.6, "Evaluate",            "Accuracy + ROC-AUC",       C["sklearn"])
box(11.5,0.75,2.4,0.55,"Predictions",        "True/False → submissions/",C["Output"])

# ── Arrows ────────────────────────────────────────────────────────────────────
for (x1,y1,x2,y2) in [
    (2.3,8.1,2.3,7.6),  (2.3,7.0,2.3,6.5),  (2.3,5.9,2.3,5.4),
    (2.3,4.8,2.3,4.3),  (2.3,3.7,2.3,3.2),  (2.3,2.6,2.3,2.1),
    (2.3,1.5,2.3,1.05), (3.6,0.75,3.4,0.75),
    (6.5,8.1,6.5,7.6),  (6.5,7.0,6.5,6.5),
    (6.5,5.9,6.5,4.3),  (5.7,0.75,6.3,0.75),
    (6.6,4.0,6.3,4.0),  (7.5,3.7,7.5,3.2),  (7.5,2.6,7.5,2.1),  (7.5,1.5,7.5,1.05),
    (8.7,0.75,10.3,2.6),(11.5,8.1,11.5,3.2),(11.5,2.6,11.5,2.1),(11.5,1.5,11.5,1.05),
]:
    arr(x1,y1,x2,y2)

legend_patches = [mpatches.Patch(color=c, label=t, alpha=0.85) for t,c in C.items()]
ax.legend(handles=legend_patches, loc="lower left", fontsize=7.5, ncol=3, framealpha=0.9)
ax.set_title("IMDB Data Pipeline — Ilesh  (colour = execution engine)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
diag_path = OUT_DIR.parent.parent / "members" / "ilesh" / "artifacts" / "pipeline_diagram.png"
diag_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(diag_path, dpi=140, bbox_inches="tight")
plt.show()
print("Diagram saved to", diag_path)

---
## 3. Step 1 — DuckDB: Data Quality Detection & Fixing

**Why DuckDB here?** In-process SQL — no cluster overhead for 8K rows.  
`TRY_CAST`, `QUANTILE_CONT`, `COPY TO PARQUET`, and UDF registration are all single SQL calls.  
Each DQ check = one SQL string → if the schema changes, edit one query.

| Fix | Decision | Reason |
|---|---|---|
| F1 Drop `Unnamed: 0` | Always exclude at read | Spurious pandas index col (range 2–9999), no information |
| F2 Drop `endYear` | **Drop, don't impute** | 90.1% missing; no row has both startYear+endYear → duration imputation impossible; near-zero signal |
| F3 `\N`→NULL | `TRY_CAST(NULLIF(NULLIF(TRIM),''),'\N')` | Handles whitespace variants + empty string + literal `\N` in one expression |
| F4 `originalTitle` fill | `COALESCE` → `primaryTitle` | 50.1% missing; best available proxy |
| F5 `numVotes` cap | IQR fence from train only | 13.8% rows above fence; cap applied to val/test using train fence → no leakage |
| F6 Drop invalid person IDs | `WHERE ... NOT IN ('\N', '\\N')` at ingestion | `\N` (297 writer rows) + `\\N` double-escaped (2 director rows); corrupt edges distort success-rate calculations |
| F7 Success-rate encoding | `INNER JOIN` train labels only | Leakage-free: val/test movies have no label → cannot affect rates |
| F8 `numVotes \N` → NULL | Implicit via `TRY_CAST(numVotes AS DOUBLE)` | Consistent with F3's explicit NULLIF; 790 rows affected |
| F9 NFKD normalisation | DuckDB UDF `normalize_title()` | ~30% of primaryTitle + ~22% of non-null originalTitle have synthetically injected accents (`Déstiny→Destiny`) |

In [ ]:
con = duckdb.connect(str(DB_PATH))

# Register Python title-normalisation as a DuckDB UDF
# Lets us call normalize_title() directly inside SQL → one-pass fix in F9
con.create_function("normalize_title", _normalize_title, [str], str)

# A) Ingestion
# all_varchar=True: reads EVERY column as text so \N tokens are preserved as
#   literal strings and not silently coerced to NaN by the parser.
# EXCLUDE column0: DuckDB names the blank-header first column "column0" (F1)
rpt(">> A) Ingesting raw data (all_varchar=True preserves \\N tokens)...")

con.execute("""
    CREATE OR REPLACE TABLE movies_train_raw AS
    SELECT * EXCLUDE (column0)
    FROM read_csv_auto(?, header=True, all_varchar=True, ignore_errors=True)
""", [TRAIN_GLOB])

for tbl, path in [("movies_val_raw", VAL_CSV), ("movies_test_raw", TEST_CSV)]:
    con.execute(f"""
        CREATE OR REPLACE TABLE {tbl} AS
        SELECT * EXCLUDE (column0)
        FROM read_csv_auto(?, header=True, all_varchar=True, ignore_errors=True)
    """, [path])

# F6: drop invalid person_id edges at ingestion
# \N  (single-escaped) — 297 writer rows, standard IMDb unknown token
# \\N (double-escaped) —   2 director rows, corrupted variant
# Both excluded; joining them would pollute success-rate encoding.
con.execute(f"""
    CREATE OR REPLACE TABLE directing AS
    SELECT tconst, director AS director_id
    FROM read_csv_auto('{DIRECTORS_CSV}', header=True)
    WHERE director IS NOT NULL
      AND TRIM(director) NOT IN ('\\N', '\\\\N')
      AND TRIM(director) != ''
""")
con.execute(f"""
    CREATE OR REPLACE TABLE writing AS
    SELECT tconst, writer AS writer_id
    FROM read_csv_auto('{WRITERS_CSV}', header=True)
    WHERE writer IS NOT NULL
      AND TRIM(writer) NOT IN ('\\N', '\\\\N')
      AND TRIM(writer) != ''
""")

n_train = con.execute("SELECT COUNT(*) FROM movies_train_raw").fetchone()[0]
rpt(f"  movies_train_raw : {n_train:,} rows")
rpt(f"  directing        : {con.execute('SELECT COUNT(*) FROM directing').fetchone()[0]:,} edges")
rpt(f"  writing          : {con.execute('SELECT COUNT(*) FROM writing').fetchone()[0]:,} edges")

In [ ]:
# B) Automatic Detection Report (D1–D13)
rpt("\n>> B) Data Quality Detection Report")

# D1: Schema validation
rpt("\n[D1] Schema validation:")
actual_cols   = set(con.execute("DESCRIBE movies_train_raw").fetchdf()["column_name"].tolist())
expected_cols = {"tconst","primaryTitle","originalTitle","startYear",
                 "endYear","runtimeMinutes","numVotes","label"}
missing_cols  = expected_cols - actual_cols
extra_cols    = actual_cols - expected_cols
rpt(f"  Expected columns present : {'✓' if not missing_cols else '✗ MISSING: ' + str(missing_cols)}")
rpt(f"  Unexpected extra columns : {'none' if not extra_cols else str(extra_cols)}")

# D2: \N token counts per VARCHAR column (incl. numVotes — TRY_CAST silently makes it NULL)
rpt("\n[D2] Disguised \\N tokens per column:")
for col in ["startYear", "endYear", "runtimeMinutes", "numVotes", "originalTitle", "primaryTitle"]:
    n = con.execute(f"SELECT COUNT(*) FROM movies_train_raw WHERE TRIM({col}) = '\\N'").fetchone()[0]
    rpt(f"  {col:<20}: {n:>5} ({100*n/n_train:.1f}%)")

# D3: True NULLs
rpt("\n[D3] True NULL counts:")
null_df = con.execute("""
    SELECT SUM(primaryTitle IS NULL) AS null_primaryTitle,
           SUM(originalTitle IS NULL) AS null_originalTitle,
           SUM(startYear IS NULL) AS null_startYear,
           SUM(numVotes IS NULL) AS null_numVotes
    FROM movies_train_raw
""").fetchdf()
print(null_df.to_string(index=False))

# D4: tconst format — every ID must match tt[0-9]+
n_bad_tconst = con.execute(
    "SELECT COUNT(*) FROM movies_train_raw WHERE NOT regexp_matches(tconst, '^tt[0-9]+$')"
).fetchone()[0]
rpt(f"\n[D4] Malformed tconst (not tt[0-9]+): {n_bad_tconst} ({'OK' if n_bad_tconst == 0 else '← FOUND'})")

# D5: Cross-split leakage — no tconst should appear in more than one split
rpt("\n[D5] Cross-split leakage (shared tconst across splits):")
for pair, q in [
    ("train ∩ val ", "SELECT COUNT(*) FROM movies_train_raw t JOIN movies_val_raw  v ON t.tconst=v.tconst"),
    ("train ∩ test", "SELECT COUNT(*) FROM movies_train_raw t JOIN movies_test_raw x ON t.tconst=x.tconst"),
    ("val ∩ test  ", "SELECT COUNT(*) FROM movies_val_raw   v JOIN movies_test_raw x ON v.tconst=x.tconst"),
]:
    n = con.execute(q).fetchone()[0]
    rpt(f"  {pair}: {n} {'(OK)' if n == 0 else '← LEAKAGE'}")

# D6: Duplicate tconst
n_dup = con.execute("SELECT COUNT(*)-COUNT(DISTINCT tconst) FROM movies_train_raw").fetchone()[0]
rpt(f"\n[D6] Duplicate tconst rows: {n_dup} ({'none' if n_dup==0 else 'FOUND'})")

# D7: numVotes IQR fence
rpt("\n[D7] numVotes IQR:")
qr = con.execute("""
    SELECT QUANTILE_CONT(TRY_CAST(numVotes AS DOUBLE),0.25) AS q1,
           QUANTILE_CONT(TRY_CAST(numVotes AS DOUBLE),0.75) AS q3
    FROM movies_train_raw WHERE numVotes IS NOT NULL
""").fetchdf()
q1 = float(qr["q1"].values[0])
q3 = float(qr["q3"].values[0])
VOTES_FENCE = round(q3 + 1.5*(q3-q1), 2)
n_above = con.execute(
    f"SELECT COUNT(*) FROM movies_train_raw WHERE TRY_CAST(numVotes AS DOUBLE) > {VOTES_FENCE}"
).fetchone()[0]
rpt(f"  Q1={q1:.0f}  Q3={q3:.0f}  fence={VOTES_FENCE:.0f}  rows_above={n_above} ({100*n_above/n_train:.1f}%)")

# D8: Referential integrity — edges vs all splits combined
rpt("\n[D8] Referential integrity (edges vs train+val+test combined):")
for etbl in ["directing", "writing"]:
    n = con.execute(f"""
        SELECT COUNT(DISTINCT e.tconst) FROM {etbl} e
        LEFT JOIN movies_train_raw t ON e.tconst=t.tconst
        LEFT JOIN movies_val_raw   v ON e.tconst=v.tconst
        LEFT JOIN movies_test_raw  x ON e.tconst=x.tconst
        WHERE t.tconst IS NULL AND v.tconst IS NULL AND x.tconst IS NULL
    """).fetchone()[0]
    rpt(f"  {etbl}: {n} edge tconst not in any split {'(OK)' if n==0 else '← ISSUE'}")

# D9: Person ID format — valid IDs must match nm[0-9]+
rpt("\n[D9] Person ID format check (pre-filter, raw CSVs):")
con.execute(f"CREATE OR REPLACE TEMP TABLE dir_raw AS SELECT * FROM read_csv_auto('{DIRECTORS_CSV}', header=True, all_varchar=True)")
con.execute(f"CREATE OR REPLACE TEMP TABLE wri_raw AS SELECT * FROM read_csv_auto('{WRITERS_CSV}',   header=True, all_varchar=True)")
for tbl, col in [("dir_raw","director"), ("wri_raw","writer")]:
    bad = con.execute(f"""
        SELECT TRIM({col}) AS bad_id, COUNT(*) AS n
        FROM {tbl}
        WHERE {col} IS NOT NULL AND NOT regexp_matches(TRIM({col}), '^nm[0-9]+$')
        GROUP BY bad_id ORDER BY n DESC
    """).fetchdf()
    if bad.empty:
        rpt(f"  {tbl}: all IDs match nm[0-9]+ (OK)")
    else:
        rpt(f"  {tbl}: {len(bad)} non-conforming pattern(s):")
        rpt("  " + bad.to_string(index=False))
rpt("  → F6 removes all non-conforming IDs at ingestion")

# D10: Label balance
rpt("\n[D10] Label balance:")
print(con.execute("""
    SELECT label, COUNT(*) AS n, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER(),2) AS pct
    FROM movies_train_raw GROUP BY label ORDER BY label
""").fetchdf().to_string(index=False))

# D11: Missingness mechanism — label rate: missing vs present
rpt("\n[D11] Missingness mechanism — label rate: missing vs present:")
for col, token in [("startYear","\\N"), ("runtimeMinutes","\\N"), ("numVotes","\\N")]:
    df = con.execute(f"""
        SELECT (TRIM({col})='{token}') AS is_missing, COUNT(*) AS n,
               ROUND(AVG(CASE WHEN label='True' THEN 1.0 ELSE 0.0 END),4) AS avg_label
        FROM movies_train_raw GROUP BY is_missing ORDER BY is_missing
    """).fetchdf()
    rpt(f"  {col}:")
    rpt("  " + df.to_string(index=False))
rpt("  (small difference → keep missingness flags as features)")

# D12: Range validity
rpt("\n[D12] Out-of-range value checks (train):")
bad_year = con.execute("""
    SELECT COUNT(*) FROM movies_train_raw
    WHERE TRY_CAST(NULLIF(TRIM(startYear),'\\N') AS INTEGER) NOT BETWEEN 1880 AND 2030
      AND TRIM(startYear) != '\\N' AND startYear IS NOT NULL
""").fetchone()[0]
bad_rt = con.execute("""
    SELECT COUNT(*) FROM movies_train_raw
    WHERE TRY_CAST(NULLIF(TRIM(runtimeMinutes),'\\N') AS INTEGER) <= 0
      AND TRIM(runtimeMinutes) != '\\N' AND runtimeMinutes IS NOT NULL
""").fetchone()[0]
bad_votes = con.execute("""
    SELECT COUNT(*) FROM movies_train_raw
    WHERE TRY_CAST(numVotes AS DOUBLE) <= 0
      AND numVotes IS NOT NULL AND TRIM(numVotes) != '\\N'
""").fetchone()[0]
rpt(f"  startYear outside [1880, 2030] : {bad_year} {'(OK)' if bad_year == 0 else '← FOUND'}")
rpt(f"  runtimeMinutes <= 0            : {bad_rt}   {'(OK)' if bad_rt == 0 else '← FOUND'}")
rpt(f"  numVotes <= 0                  : {bad_votes} {'(OK)' if bad_votes == 0 else '← FOUND'}")

# D13: Text corruption — both title columns
rpt("\n[D13] Text corruption — non-ASCII titles:")
primary_df  = con.execute("SELECT primaryTitle FROM movies_train_raw").fetchdf()
original_df = con.execute("SELECT originalTitle FROM movies_train_raw WHERE originalTitle IS NOT NULL").fetchdf()
n_primary  = primary_df["primaryTitle"].apply(lambda x: _normalize_title(str(x)) != str(x)).sum()
n_original = original_df["originalTitle"].apply(lambda x: _normalize_title(str(x)) != str(x)).sum()
rpt(f"  primaryTitle  rows with synthetic accents: {n_primary}  ({100*n_primary/n_train:.1f}%)")
rpt(f"  originalTitle rows with synthetic accents: {n_original} ({100*n_original/len(original_df):.1f}% of non-null rows)")

In [ ]:
# C) Automatic Fixes
rpt("\n>> C) Applying fixes...")

def _build_clean(out_tbl, src_expr, has_label):
    """One-pass clean applying F2, F3, F4, F9 simultaneously."""
    label_col = ", CAST(label AS BOOLEAN) AS label" if has_label else ""
    con.execute(f"""
        CREATE OR REPLACE TABLE {out_tbl} AS
        SELECT
            tconst,
            -- F9: normalize synthetic accent injection via DuckDB UDF
            normalize_title(primaryTitle) AS primaryTitle,
            -- F4+F9: originalTitle NULLs → fallback to normalized primaryTitle
            COALESCE(
                NULLIF(normalize_title(TRIM(COALESCE(originalTitle,''))), ''),
                normalize_title(primaryTitle)
            ) AS originalTitle,
            -- F3: '\\N' / '' / whitespace → NULL, then safe integer cast
            TRY_CAST(NULLIF(NULLIF(TRIM(startYear),''),'\\N') AS INTEGER) AS startYear,
            -- F2: endYear DROPPED (90.1% missing, no row has both startYear+endYear)
            TRY_CAST(NULLIF(NULLIF(TRIM(runtimeMinutes),''),'\\N') AS INTEGER) AS runtimeMinutes,
            TRY_CAST(numVotes AS DOUBLE) AS numVotes
            {label_col}
        FROM ({src_expr})
    """)

_build_clean("movies_train_clean", "SELECT * FROM movies_train_raw", has_label=True)
_build_clean("movies_val_clean",   "SELECT * FROM movies_val_raw",   has_label=False)
_build_clean("movies_test_clean",  "SELECT * FROM movies_test_raw",  has_label=False)

# F5: numVotes IQR cap — train fence applied to all splits (no leakage)
for tbl in ["movies_train_clean", "movies_val_clean", "movies_test_clean"]:
    con.execute(f"""
        CREATE OR REPLACE TABLE {tbl} AS
        SELECT * REPLACE (
            CASE WHEN numVotes > {VOTES_FENCE} THEN {VOTES_FENCE}
                 ELSE numVotes END AS numVotes
        ) FROM {tbl}
    """)
rpt(f"[F5] numVotes capped at {VOTES_FENCE:.0f} ({n_above} train rows)")

# F7: success-rate target encoding (train labels only — INNER JOIN prevents leakage)
con.execute("""
    CREATE OR REPLACE TABLE director_success AS
    SELECT d.director_id,
           COUNT(DISTINCT d.tconst)     AS director_movie_count,
           AVG(CAST(m.label AS DOUBLE)) AS director_success_rate
    FROM directing d
    INNER JOIN movies_train_clean m ON d.tconst = m.tconst
    GROUP BY d.director_id
""")
con.execute("""
    CREATE OR REPLACE TABLE writer_success AS
    SELECT w.writer_id,
           COUNT(DISTINCT w.tconst)     AS writer_movie_count,
           AVG(CAST(m.label AS DOUBLE)) AS writer_success_rate
    FROM writing w
    INNER JOIN movies_train_clean m ON w.tconst = m.tconst
    GROUP BY w.writer_id
""")
rpt(f"[F7] {con.execute('SELECT COUNT(*) FROM director_success').fetchone()[0]:,} directors encoded")
rpt(f"[F7] {con.execute('SELECT COUNT(*) FROM writer_success').fetchone()[0]:,} writers encoded")

In [ ]:
# D) Post-fix verification
rpt("\n>> D) Post-fix verification:")
v = con.execute("""
    SELECT COUNT(*) AS total,
           SUM(startYear IS NULL) AS null_startYear,
           SUM(runtimeMinutes IS NULL) AS null_runtimeMinutes,
           SUM(numVotes IS NULL) AS null_numVotes,
           SUM(primaryTitle IS NULL) AS null_primaryTitle,
           MAX(numVotes) AS max_numVotes_capped
    FROM movies_train_clean
""").fetchdf()
print(v.to_string(index=False))

# E) Export Parquet for PySpark
rpt("\n>> E) Exporting Parquet...")
exports = {
    "movies_train_clean" : PQDIR/"movies_train.parquet",
    "movies_val_clean"   : PQDIR/"movies_val.parquet",
    "movies_test_clean"  : PQDIR/"movies_test.parquet",
    "directing"          : PQDIR/"directing.parquet",
    "writing"            : PQDIR/"writing.parquet",
    "director_success"   : PQDIR/"director_success.parquet",
    "writer_success"     : PQDIR/"writer_success.parquet",
}
for tbl, path in exports.items():
    con.execute(f"COPY {tbl} TO '{path}' (FORMAT PARQUET)")
    n = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    rpt(f"  [OK] {tbl:<25}: {n:,} rows → {path.name}")

con.close()
RPT_PATH.write_text("\n".join(_log))
print(f"\nStep 1 complete → {PQDIR}")

---
## 4. Step 2 — PySpark: Imputation, Feature Engineering & Scaling

**Why PySpark here?**  
After DuckDB's quality fixes, NULLs remain in `startYear` (~10%), `runtimeMinutes` (<1%), `numVotes` (~10%).  
PySpark MLlib `Imputer` is a **Pipeline stage**: fit on train only, transform val/test with the same medians — the standard leakage-free distributed pattern from the course.  
The `VectorAssembler + StandardScaler` pipeline is **serialisable**: re-running on new data only requires calling `pipeline.transform()`, not rewriting logic.

**Feature decisions**:
| Feature | Signal | Why |
|---|---|---|
| `startYear` | r=−0.264 | Older = more likely classic (True) |
| `runtimeMinutes` | r=+0.302 | Strongest single feature |
| `log_numVotes` | r=+0.246 | log1p reduces skewness 9.81→1.22; linearises relationship for LR |
| `avg/max_director_success_rate` | — | Train-only target encoding; 0.5 neutral prior for unknown directors |
| `avg/max_writer_success_rate` | — | Same |
| `is_missing_numVotes` | slight | Label rate 0.478 missing vs 0.504 present |
| `is_missing_startYear` | slight | Label rate 0.477 missing vs 0.504 present |

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml import Pipeline as SparkPipeline
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler

spark = (
    SparkSession.builder
    .appName("IMDB-Ilesh-Step2")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

In [ ]:
movies_train = spark.read.parquet(str(PQDIR/"movies_train.parquet"))
movies_val   = spark.read.parquet(str(PQDIR/"movies_val.parquet"))
movies_test  = spark.read.parquet(str(PQDIR/"movies_test.parquet"))
directing    = spark.read.parquet(str(PQDIR/"directing.parquet"))
writing      = spark.read.parquet(str(PQDIR/"writing.parquet"))
dir_success  = spark.read.parquet(str(PQDIR/"director_success.parquet"))
wri_success  = spark.read.parquet(str(PQDIR/"writer_success.parquet"))

for name, sdf in [("train",movies_train),("val",movies_val),("test",movies_test)]:
    print(f"  {name}: {sdf.count():,} rows")

In [ ]:
# B) Missingness flags — added BEFORE imputation so they capture actual NULL pattern
def _add_flags(sdf):
    return (sdf
            .withColumn("is_missing_numVotes",
                        F.when(F.col("numVotes").isNull(),  1.0).otherwise(0.0))
            .withColumn("is_missing_startYear",
                        F.when(F.col("startYear").isNull(), 1.0).otherwise(0.0))
            .withColumn("is_missing_runtimeMinutes",
                        F.when(F.col("runtimeMinutes").isNull(), 1.0).otherwise(0.0)))

movies_train = _add_flags(movies_train)
movies_val   = _add_flags(movies_val)
movies_test  = _add_flags(movies_test)

# C) M:M LEFT JOIN — director/writer aggregate features
# LEFT JOIN: every movie kept even with no person data
# Unknown persons in val/test → NULL rate → imputed to 0.5 (neutral prior)
def _add_person_features(sdf):
    dir_agg = (
        directing.join(dir_success, on="director_id", how="left")
        .groupBy("tconst")
        .agg(
            F.countDistinct("director_id").cast(DoubleType()).alias("num_directors"),
            F.avg("director_success_rate").alias("avg_director_success_rate"),
            F.max("director_success_rate").alias("max_director_success_rate"),
        )
    )
    wri_agg = (
        writing.join(wri_success, on="writer_id", how="left")
        .groupBy("tconst")
        .agg(
            F.countDistinct("writer_id").cast(DoubleType()).alias("num_writers"),
            F.avg("writer_success_rate").alias("avg_writer_success_rate"),
            F.max("writer_success_rate").alias("max_writer_success_rate"),
        )
    )
    return sdf.join(dir_agg, on="tconst", how="left").join(wri_agg, on="tconst", how="left")

movies_train = _add_person_features(movies_train)
movies_val   = _add_person_features(movies_val)
movies_test  = _add_person_features(movies_test)

# D) Cast to Double (MLlib requirement)
NUMERIC_COLS = ["startYear","runtimeMinutes","numVotes",
                "num_directors","num_writers",
                "avg_director_success_rate","avg_writer_success_rate",
                "max_director_success_rate","max_writer_success_rate"]

def _cast_double(sdf):
    for col in NUMERIC_COLS:
        sdf = sdf.withColumn(col, F.col(col).cast(DoubleType()))
    return sdf

movies_train = _cast_double(movies_train)
movies_val   = _cast_double(movies_val)
movies_test  = _cast_double(movies_test)
print("Flags, M:M joins, and casts done.")

In [ ]:
# E) NULL report before imputation
print("NULLs before imputation (train):")
total = movies_train.count()
nc = movies_train.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in NUMERIC_COLS]
).toPandas()
for col in NUMERIC_COLS:
    n = int(nc[col].values[0])
    if n > 0: print(f"  {col:<35}: {n} ({100*n/total:.1f}%)")

# F) Imputation
# Zero: num_directors/writers — factually correct for movies with no edges
# 0.5:  success rates for unknown persons — neutral prior (no bias either direction)
# Median (MLlib Imputer, FIT ON TRAIN ONLY): startYear, runtimeMinutes, numVotes
ZERO_FILL    = ["num_directors","num_writers"]
NEUTRAL_FILL = ["avg_director_success_rate","avg_writer_success_rate",
                "max_director_success_rate","max_writer_success_rate"]

def _fixed_fills(sdf):
    for col in ZERO_FILL:    sdf = sdf.withColumn(col, F.coalesce(F.col(col), F.lit(0.0)))
    for col in NEUTRAL_FILL: sdf = sdf.withColumn(col, F.coalesce(F.col(col), F.lit(0.5)))
    return sdf

movies_train = _fixed_fills(movies_train)
movies_val   = _fixed_fills(movies_val)
movies_test  = _fixed_fills(movies_test)

MEDIAN_COLS   = ["startYear","runtimeMinutes","numVotes"]
imputer       = Imputer(strategy="median",
                        inputCols=MEDIAN_COLS,
                        outputCols=[c+"_imp" for c in MEDIAN_COLS])
imputer_model = imputer.fit(movies_train)   # ← FIT ON TRAIN ONLY

def _apply_imputer(sdf):
    sdf = imputer_model.transform(sdf)
    for col in MEDIAN_COLS:
        sdf = sdf.drop(col).withColumnRenamed(col+"_imp", col)
    return sdf

movies_train = _apply_imputer(movies_train)
movies_val   = _apply_imputer(movies_val)
movies_test  = _apply_imputer(movies_test)
print("Imputation complete. Remaining NULLs:",
      sum(int(movies_train.select(F.sum(F.col(c).isNull().cast("int"))).collect()[0][0])
          for c in MEDIAN_COLS))

In [ ]:
# G) Feature engineering + VectorAssembler + StandardScaler
# log1p(numVotes): skewness 9.81 → 1.22; True movies average 5× more votes
# → log linearises this relationship for LR/SVM
def _engineer(sdf):
    return sdf.withColumn("log_numVotes", F.log1p(F.col("numVotes"))).drop("numVotes")

movies_train = _engineer(movies_train)
movies_val   = _engineer(movies_val)
movies_test  = _engineer(movies_test)

FEATURE_COLS = [
    "startYear", "runtimeMinutes", "log_numVotes",
    "num_directors", "num_writers",
    "avg_director_success_rate", "max_director_success_rate",
    "avg_writer_success_rate",   "max_writer_success_rate",
    "is_missing_numVotes", "is_missing_startYear", "is_missing_runtimeMinutes",
]

# Pipeline: VectorAssembler + StandardScaler — FIT ON TRAIN ONLY
# withMean=True, withStd=True: critical for LR (gradient descent is scale-sensitive)
assembler  = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features_raw", handleInvalid="keep")
scaler     = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)
prep_model = SparkPipeline(stages=[assembler, scaler]).fit(movies_train)  # FIT ON TRAIN ONLY

movies_train = prep_model.transform(movies_train)
movies_val   = prep_model.transform(movies_val)
movies_test  = prep_model.transform(movies_test)
print("VectorAssembler + StandardScaler pipeline fitted and applied.")

In [ ]:
# H) Export flat CSVs for sklearn Step 3
def _export(sdf, path, has_label):
    keep = ["tconst"] + (["label"] if has_label else [])
    pdf  = sdf.select(keep + ["features"]).toPandas()
    import numpy as np
    mat  = np.array([v.toArray() for v in pdf["features"]])
    out  = pd.concat([pdf[keep].reset_index(drop=True),
                      pd.DataFrame(mat, columns=FEATURE_COLS)], axis=1)
    out.to_csv(path, index=False)
    print(f"  [OK] {Path(path).name}: {out.shape}  NULLs={out.isnull().sum().sum()}")
    return out

train_pdf = _export(movies_train, FEAT_DIR/"train_features.csv", has_label=True)
val_pdf   = _export(movies_val,   FEAT_DIR/"val_features.csv",   has_label=False)
test_pdf  = _export(movies_test,  FEAT_DIR/"test_features.csv",  has_label=False)

spark.stop()
print(f"Step 2 complete → {FEAT_DIR}")

---
## 5. Step 3 — sklearn: Model Training & Predictions

**Why sklearn here (not PySpark MLlib or DuckDB)?**  
Features are already scaled by the Spark pipeline — no extra preprocessing.  
`LogisticRegression` is fast, interpretable, and well-suited to the small (8K row) feature matrix.  
PySpark MLlib would add JVM overhead with no benefit at this data size.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
import glob as _glob
import numpy as np

avail = [c for c in FEATURE_COLS if c in train_pdf.columns]
X_all = train_pdf[avail].fillna(0).astype(float)
y_all = (train_pdf["label"].astype(str).str.strip() == "True").astype(int)

X_tr, _, y_tr, _ = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
print(f"Train={len(X_tr):,}  features={len(avail)}")

# Features already StandardScaled by Spark pipeline — no extra scaler needed
model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
model.fit(X_tr, y_tr)

# ── Leakage-free baseline comparison ──────────────────────────────────────────
# IQR fence, medians, success rates all computed on the 80% fold only,
# then applied to the 20% holdout → no label leakage.
print("\n--- Leakage-free baseline comparison (fold-aware pipeline) ---")

raw_frames = [pd.read_csv(p) for p in sorted(_glob.glob(str(RAW_CSV / "train-*.csv")))]
raw_df = pd.concat(raw_frames, ignore_index=True)

movies_raw = pd.DataFrame({
    "tconst":         raw_df["tconst"],
    "startYear":      pd.to_numeric(raw_df["startYear"].replace("\\N", float("nan")), errors="coerce"),
    "runtimeMinutes": pd.to_numeric(raw_df["runtimeMinutes"].replace("\\N", float("nan")), errors="coerce"),
    "numVotes":       pd.to_numeric(raw_df["numVotes"].replace("\\N", float("nan")), errors="coerce"),
    "label":          (raw_df["label"].astype(str).str.strip() == "True").astype(int),
    "primaryTitle":   raw_df["primaryTitle"].fillna(""),
})

dir_df = pd.read_csv(str(RAW_CSV / "movie_directors.csv"))
wri_df = pd.read_csv(str(RAW_CSV / "movie_writers.csv"))
dir_df = dir_df[~dir_df["director"].isin(["\\N","\\\\N",""])].dropna(subset=["director"])
wri_df = wri_df[~wri_df["writer"].isin(["\\N","\\\\N",""])].dropna(subset=["writer"])
dir_df.columns = ["tconst","director_id"]
wri_df.columns = ["tconst","writer_id"]

tr_idx, hv_idx = train_test_split(
    movies_raw.index, test_size=0.2, random_state=42, stratify=movies_raw["label"]
)
tr = movies_raw.loc[tr_idx].copy().reset_index(drop=True)
hv = movies_raw.loc[hv_idx].copy().reset_index(drop=True)

# (A) Naive: majority class
maj_acc = accuracy_score(hv["label"], np.full(len(hv), int(tr["label"].mode()[0])))

# (B) Naive: title length only (mirrors shared baseline in src/)
tl_tr = tr["primaryTitle"].str.len().values.reshape(-1,1).astype(float)
tl_hv = hv["primaryTitle"].str.len().values.reshape(-1,1).astype(float)
naive_lr  = LogisticRegression(C=1.0, max_iter=1000, random_state=42).fit(tl_tr, tr["label"])
naive_acc = accuracy_score(hv["label"], naive_lr.predict(tl_hv))
naive_auc = roc_auc_score(hv["label"], naive_lr.predict_proba(tl_hv)[:,1])

# (C) Naive: raw 3 numerics, no cleaning, no engineering
rn_tr = tr[["numVotes","runtimeMinutes","startYear"]].fillna(0).astype(float)
rn_hv = hv[["numVotes","runtimeMinutes","startYear"]].fillna(0).astype(float)
raw_lr  = LogisticRegression(C=1.0, max_iter=1000, random_state=42).fit(rn_tr, tr["label"])
raw_acc = accuracy_score(hv["label"], raw_lr.predict(rn_hv))
raw_auc = roc_auc_score(hv["label"], raw_lr.predict_proba(rn_hv)[:,1])

# (D) Full pipeline — fold-aware, no leakage
q1f, q3f = tr["numVotes"].quantile(0.25), tr["numVotes"].quantile(0.75)
fence_f  = q3f + 1.5*(q3f-q1f)
meds = {c: tr[c].median() for c in ["startYear","runtimeMinutes","numVotes"]}
for df in [tr, hv]:
    df["numVotes"] = df["numVotes"].clip(upper=fence_f)
    df["is_missing_numVotes"]       = df["numVotes"].isna().astype(float)
    df["is_missing_startYear"]      = df["startYear"].isna().astype(float)
    df["is_missing_runtimeMinutes"] = df["runtimeMinutes"].isna().astype(float)
    for c, v in meds.items():
        df[c] = df[c].fillna(v)
    df["log_numVotes"] = np.log1p(df["numVotes"])

def _add_person_feats(tr_df, apply_df, edges, id_col, prefix):
    rates = (edges.merge(tr_df[["tconst","label"]], on="tconst", how="inner")
                  .groupby(id_col)["label"].mean().rename("rate").reset_index())
    agg   = (edges.merge(rates, on=id_col, how="left")
                  .groupby("tconst")["rate"]
                  .agg(avg="mean", max="max").reset_index()
                  .rename(columns={"avg":f"avg_{prefix}_success_rate",
                                   "max":f"max_{prefix}_success_rate"}))
    cnt   = (edges.groupby("tconst")[id_col].nunique()
                  .rename(f"num_{prefix}s").reset_index())
    out = apply_df.merge(cnt, on="tconst", how="left").merge(agg, on="tconst", how="left")
    out[f"num_{prefix}s"]             = out[f"num_{prefix}s"].fillna(0)
    out[f"avg_{prefix}_success_rate"] = out[f"avg_{prefix}_success_rate"].fillna(0.5)
    out[f"max_{prefix}_success_rate"] = out[f"max_{prefix}_success_rate"].fillna(0.5)
    return out

tr = _add_person_feats(tr, tr, dir_df, "director_id", "director")
hv = _add_person_feats(tr, hv, dir_df, "director_id", "director")
tr = _add_person_feats(tr, tr, wri_df, "writer_id",   "writer")
hv = _add_person_feats(tr, hv, wri_df, "writer_id",   "writer")

PIPE_FEAT = ["startYear","runtimeMinutes","log_numVotes",
             "num_directors","num_writers",
             "avg_director_success_rate","max_director_success_rate",
             "avg_writer_success_rate","max_writer_success_rate",
             "is_missing_numVotes","is_missing_startYear","is_missing_runtimeMinutes"]
pipe_lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
pipe_lr.fit(tr[PIPE_FEAT].fillna(0).astype(float), tr["label"])
pipe_acc = accuracy_score(hv["label"], pipe_lr.predict(hv[PIPE_FEAT].fillna(0).astype(float)))
pipe_auc = roc_auc_score(hv["label"], pipe_lr.predict_proba(hv[PIPE_FEAT].fillna(0).astype(float))[:,1])

print(f"  {'Model':<44} {'Accuracy':>9} {'ROC-AUC':>9}")
print(f"  {'-'*62}")
print(f"  {'Majority class (always predict True/False)':<44} {maj_acc:>9.4f} {'—':>9}")
print(f"  {'Naive: title length only (shared baseline)':<44} {naive_acc:>9.4f} {naive_auc:>9.4f}")
print(f"  {'Raw numerics (no cleaning, no engineering)':<44} {raw_acc:>9.4f} {raw_auc:>9.4f}")
print(f"  {'Ilesh pipeline — leakage-free (12 features)':<44} {pipe_acc:>9.4f} {pipe_auc:>9.4f}")
print(f"  {'-'*62}")
print(f"  Pipeline gain over naive : +{pipe_acc-naive_acc:.4f} accuracy  +{pipe_auc-naive_auc:.4f} AUC")

In [ ]:
import datetime
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# IMPORTANT: Spark's toPandas() does not preserve row order.
# Kaggle expects predictions in the same order as the original CSV.
# Re-align by merging on tconst with the original CSVs before writing.
def _write_sub(X, raw_csv_path, path):
    raw_order = pd.read_csv(raw_csv_path)[["tconst"]]
    X_ordered = raw_order.merge(X, on="tconst", how="left")
    preds = model.predict(X_ordered[avail].fillna(0).astype(float))
    lines = ["True" if p == 1 else "False" for p in preds]
    path.write_text("\n".join(lines))
    print(f"  {path.name}: {len(lines)} rows (order aligned to original CSV)")

_write_sub(val_pdf,  VAL_CSV,  SUB_DIR / f"ilesh_val_{ts}.txt")
_write_sub(test_pdf, TEST_CSV, SUB_DIR / f"ilesh_test_{ts}.txt")
print(f"Submissions written → {SUB_DIR}")

---
## 6. Reusability & Schema Evolution

### How reusable is this pipeline?

| Design decision | Benefit |
|---|---|
| All paths resolved from `PROJECT_ROOT` | Works locally, in Docker, on any machine without editing paths |
| `VOTES_FENCE` computed once from train, applied to val/test | No data leakage; re-run on new data just re-fits |
| MLlib `Imputer` + `StandardScaler` fitted on train only | Serialisable pipeline object; call `.transform()` on any new split |
| DuckDB UDF for title normalisation | Runs inside SQL; no Pandas loop needed; easy to extend |
| Success-rate encoding via `INNER JOIN` on train | Leakage-proof; new splits use same rates without re-fitting |
| `FEATURE_COLS` declared in one place | Add/remove features by editing one list |
| `pipeline.py` script mirrors the notebook | Run headless in Docker without Jupyter |

### How much effort if the schema changes?

| Change | Effort | Where |
|---|---|---|
| New column added to CSV | **Trivial** | Add to `FEATURE_COLS` if numeric, or add a SQL fix in `_build_clean()` |
| Column renamed | **Low** | Grep for old name; update in `_build_clean()` and `FEATURE_COLS` |
| New disguised-missing token | **Trivial** | Add to `NULLIF` chain in `_build_clean()` SQL |
| New CSV data source | **Low** | Add `read_csv_auto` call and optional DuckDB table in Step 1 |
| Dataset grows 100× | **No code change** | DuckDB scales in-process; Step 2 moves to Spark cluster by changing `.master()` |

### DuckDB vs PySpark — decision criteria used

Three questions from the course:
1. **Relational / SQL-natural?** → DuckDB. Joins, group-bys, `TRY_CAST`, `QUANTILE_CONT`, and `COPY TO PARQUET` are idiomatic SQL — far cleaner than Pandas chains.
2. **Needs to be leakage-free and reusable across splits?** → PySpark MLlib Pipeline. `Imputer.fit()` + `.transform()` is the standard distributed fit/transform paradigm; serialisable and cluster-ready.
3. **Small, cheap, already in memory?** → sklearn. LR on a 7959×11 matrix is instant; no Spark or DuckDB overhead justified.